# 01 — Build the research data

**What this does:** pulls historical prices, fundamentals, factors, benchmark returns, and the live-price extension, then builds the point-in-time research panel.

**Run first:** complete the setup in `START_HERE.ipynb` and check the settings in `config.py`. Nothing else must be run before this notebook.

**Produces:** source files under `data/` and the configured panel, normally `data/panel_russell3000.parquet`.

**Time:** allow 30 minutes to two hours for the first WRDS build and roughly 5–20 minutes from cache. Run one cell at a time and read its confirmation before continuing.

## Step 1 — Load the project and review settings

Set `REQUERY_ALL_WRDS_SOURCES` to `True` for the first build or when you deliberately want every pull to bypass its cache. Leave it `False` for a fast cache-based development rebuild. The cell prints the selected mode prominently.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = next(
    (path for path in (Path.cwd(), *Path.cwd().parents) if (path / 'config.py').is_file()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError('Could not find the project folder containing config.py.')
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from IPython.display import display
import importlib
import config
import src.notebook_support as notebook_support

# Reload config first so a long-running kernel sees newly added settings.
config = importlib.reload(config)
notebook_support = importlib.reload(notebook_support)
settings_table = notebook_support.settings_table

REQUERY_ALL_WRDS_SOURCES = False  # True = bypass every WRDS cache in this notebook.
display(settings_table())
print(f'Project loaded from: {PROJECT_ROOT}')
build_mode = (
    'FRESH WRDS MODE — every source will be queried again'
    if REQUERY_ALL_WRDS_SOURCES
    else 'CACHE MODE — existing raw-query caches will be reused when available'
)
print(f'DATA BUILD MODE: {build_mode}')

## Step 2 — Pull CRSP prices and short interest

This creates `data/prices.parquet` and `data/short_interest.parquet`. The confirmation should show more than one million price rows, no duplicate security-months, and history beginning by 2003. A fresh WRDS query can be the longest step.

In [ ]:
from src.data import pull_prices
from src.notebook_support import confirm_prices

prices = pull_prices.main(refresh_from_wrds=REQUERY_ALL_WRDS_SOURCES)
display(confirm_prices(prices))

## Step 3 — Pull factors and benchmark returns

This creates `data/factors.parquet`, including the market, size, value, profitability, investment, momentum, risk-free, S&P 500, and Russell 3000 series. Required factor columns and duplicate months are checked explicitly.

In [ ]:
from src.data import pull_factors
from src.notebook_support import confirm_factors

factors = pull_factors.main(refresh_from_wrds=REQUERY_ALL_WRDS_SOURCES)
display(confirm_factors(factors))

## Step 4 — Pull Compustat fundamentals and reference histories

This creates `data/fundamentals.parquet`, `data/link.parquet`, and `data/sectors.parquet`. The confirmation should show both annual and quarterly observations, more than 20,000 companies, and no duplicate company-date-frequency rows.

In [ ]:
from src.data import pull_fundamentals
from src.notebook_support import confirm_fundamentals

fundamentals = pull_fundamentals.main(refresh_from_wrds=REQUERY_ALL_WRDS_SOURCES)
display(confirm_fundamentals(fundamentals))

## Step 5 — Pull the provisional live-price extension

This creates `data/live_prices.parquet` for months beyond the CRSP cutoff. It flags, but does not delete, liquid securities with extreme returns. The latest live month must be within the staleness limit shown in `config.py`.

In [ ]:
from src.data import pull_live_prices
from src.notebook_support import confirm_live_prices

live_prices = pull_live_prices.main(refresh_from_wrds=REQUERY_ALL_WRDS_SOURCES)
display(confirm_live_prices(live_prices))

## Step 6 — Build and validate the point-in-time panel

This joins prices, company links, fundamentals, short interest, and sectors using their date-valid rules. It writes the configured panel and prints the date a subsequent rebalance would use as its signal date. A blocking failure stops the build; do not bypass one. Warnings are printed for review.

In [ ]:
from src.data import build_panel
from src.notebook_support import announce_rebalance_signal_date, confirm_panel

panel, checks = build_panel.main()
display(confirm_panel(panel, checks))
rebalance_signal_date = announce_rebalance_signal_date(panel)

## Step 7 — Review panel health

These are presentation views of the panel builder's existing checks: a sample of rows, recent investable-name counts, and recent fundamental and sector coverage. Investable Russell names should normally be in the low thousands, and coverage should not collapse suddenly in the newest month.

In [ ]:
display(build_panel.panel_preview(panel))
display(build_panel.investable_names_by_month(panel).tail(12))
display(build_panel.coverage_by_year(panel, population='investable').tail(5))
print('Panel review displayed: sample rows, latest 12 monthly name counts, and latest 5 annual coverage rows.')

## What was produced and what comes next

You should now have historical prices, short interest, fundamentals, company links, sectors, factors, live prices, and the configured research panel under `data/`. The panel confirmation above must show zero blocking failures.

- For the normal monthly process, continue to **02_monthly_update.ipynb**.
- To prepare the monthly target portfolio and trade list, use **02_monthly_update.ipynb**.
- To evaluate signals, continue to **03_signal_research.ipynb**.
- To run the model and historical portfolio, continue to **04_backtest.ipynb**.

## If something fails

- **WRDS login or timeout:** confirm the username shown in Step 1 and test the WRDS website. Retry once; then contact the WRDS administrator.
- **Missing file:** run the earlier step that creates the named file, or restart this notebook from the top.
- **Live-price staleness error:** set `REQUERY_ALL_WRDS_SOURCES = True` and rerun from Step 1; do not weaken the guard.
- **Panel check failure:** stop, save the full check table, and investigate the failing check.
- **Undefined variable or out-of-order cell:** restart the kernel and run all cells from the top. See `START_HERE.ipynb` for the full troubleshooting table.